# Week 1 · Day 1 — Explaining Aviatrix Security with GPT

**Course:** AI Engineer Core Track (LLM Engineering) — a personal project built on Ed Donner's Day 1 lab.

**Goal:** use GPT as a senior network and cloud security engineer to explain the [Aviatrix documentation](https://docs.aviatrix.com/), with a deep dive into how Aviatrix secures cloud traffic.

The pattern is the same as in Ed's lab:

```
documentation pages → text → messages (system + user) → GPT → explanation in Markdown
```

| Part | What it does |
| --- | --- |
| **1. Quick overview** | Summarizes the documentation home page as a networking expert (BGP, VPN, cloud networking) |
| **2. Security deep dive** | Reads 8 security pages at once and explains firewall, egress, SmartGroups, WebGroups and TLS inspection |
| **3. Bonus** | Uses `llms.txt` to map the whole documentation and suggest a reading path |

> **Before you start** — You need the `.venv` environment, the libraries and the `.env` file with your API key.
> If you haven't set them up yet, follow section 0 of [`day1_en.ipynb`](day1_en.ipynb). Then run the cells **from top to bottom** with Shift+Enter.

## Setup

### Imports

| Import | What it's for |
| --- | --- |
| `requests` | Downloading pages |
| `BeautifulSoup` | Extracting text from HTML pages |
| `load_dotenv`, `os` | Reading the API key from `.env` |
| `OpenAI` | Calling GPT |
| `Markdown`, `display` | Showing the answer formatted |

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

### API key and client

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key found: check your .env file")
elif not api_key.startswith("sk-proj-"):
    print("The key doesn't start with sk-proj-: make sure it's the right one")
elif api_key.strip() != api_key:
    print("The key has spaces at the start or end: remove them from .env")
else:
    print("API key found!")

openai = OpenAI()
MODEL = "gpt-4.1-mini"

> **Check** — It should print `API key found!`.

### Helper functions

Two ways to read documentation:

- **`fetch_website_contents`** reads a normal HTML page, like Ed's scraper. It also strips `nav`, `header` and `footer`, which take up most of the text on documentation sites, and keeps up to 5,000 characters instead of 2,000.
- **`fetch_markdown`** reads a page in **Markdown format**. The Aviatrix docs publish every page as Markdown too: just add `.md` to the address. The text arrives already clean, so BeautifulSoup isn't needed.
- **`collect_pages`** joins several Markdown pages into one text. Each page gets a `### SOURCE: <title> (<url>)` label, so the model knows where each fact comes from and can cite it. A page that fails is skipped instead of stopping everything.

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url, max_chars=5_000):
    """Return title + readable text of an HTML page, truncated to max_chars."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string.strip() if soup.title and soup.title.string else "No title"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input", "nav", "header", "footer", "svg"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:max_chars]


def fetch_markdown(url, max_chars=6_000):
    """Download a .md documentation page and truncate it to max_chars."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.text[:max_chars]


def collect_pages(pages):
    """Join several pages into one text, each labelled with its title and URL."""
    parts = []
    for title, url in pages.items():
        try:
            parts.append(f"### SOURCE: {title} ({url})\n\n{fetch_markdown(url)}")
            print(f"OK   {title}")
        except requests.RequestException as e:
            print(f"SKIP {title}: {e}")
    return "\n\n---\n\n".join(parts)

## Part 1 — Quick overview of the documentation

First, a technical summary of the home page, written by a networking expert.

### Look at the raw text first

Before calling GPT, check what the scraper sees. If there's little or nothing here, the problem is the scraper, not the model.

In [ ]:
HOME_URL = "https://docs.aviatrix.com/"

home = fetch_website_contents(HOME_URL)
print(f"Characters extracted: {len(home)}\n")
print(home)

### The networking expert prompt

| Choice | Effect |
| --- | --- |
| Precise role with the technologies listed | Steers the model toward a specialist's vocabulary |
| "for other network engineers" | No dumbing down |
| Example terminology (ASN, AS-path, ECMP, IKE phase 1/2) | Encourages those terms where the text supports them |
| "Never invent configuration values" | The most important rule for technical topics |
| Fixed structure | Summaries are always ordered and comparable across pages |
| "Not covered on this page" | Stops the model from inventing BGP or VPN details |

In [ ]:
network_system_prompt = """
You are a senior network engineer with deep expertise in enterprise networking,
multi-cloud networking (AWS, Azure, GCP), BGP routing and VPN technologies
(IPsec, IKEv2, SSL/TLS VPN). You read the contents of a technical web page
and write a precise, technical summary for other network engineers.

Rules:
- Ignore navigation menus, footers, cookie banners and marketing fluff.
- Use correct technical terminology (e.g. ASN, route propagation, AS-path,
  local preference, ECMP, IKE phase 1/2, tunnel interfaces, CIDR, transit gateway).
- Be factual: only state what the page says. Never invent configuration values,
  versions or features. If something important is missing or unclear, say so.
- Be concise: prefer short bullet points over long paragraphs.

Structure your answer with these sections:

## Overview
Two or three sentences: what the page or product is and what problem it solves.

## Key technical concepts
The main technologies, protocols and components mentioned, one line each.

## How it works
The architecture or data flow in a few bullets (routing, tunnels, segmentation, failover).

## BGP and VPN details
Any routing or VPN specifics: ASNs, route advertisement, attributes, encryption,
redundancy. Write "Not covered on this page" if there are none.

## Operational notes
Prerequisites, limitations, best practices or gotchas worth knowing.

## Takeaway
One sentence on why this matters to a network engineer.

Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

network_user_prompt_prefix = """
Here are the contents of a technical web page.
Write a technical summary following the required structure.

"""

### Summarize

In [ ]:
def messages_for(system_prompt, user_prompt_prefix, content):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + content},
    ]


def explain(system_prompt, user_prompt_prefix, content):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages_for(system_prompt, user_prompt_prefix, content),
    )
    display(Markdown(response.choices[0].message.content))


explain(network_system_prompt, network_user_prompt_prefix, home)

> **Tip** — The home page is mostly an index, so *How it works* and *BGP and VPN details* may come back almost empty. That's expected: the real content is in the specific pages, which is why Part 2 reads them directly.
>
> To summarize any other page, pass it to `fetch_website_contents`, for example:
> `explain(network_system_prompt, network_user_prompt_prefix, fetch_website_contents("https://docs.aviatrix.com/..."))`

## Part 2 — Security deep dive

To explain Aviatrix security properly, the home page isn't enough: the information lives in specific pages.
These addresses come from the site's [`llms.txt`](https://docs.aviatrix.com/llms.txt) index. Each page is short (about 1,800 characters), so we **join several** and have GPT explain them together in one answer.

### 2.1 The security pages

In [ ]:
BASE = "https://docs.aviatrix.com/docs/cloud"

SECURITY_PAGES = {
    "Security Overview": f"{BASE}/security/index.md",
    "About Egress Security": f"{BASE}/security/egress-nat/about-egress-security.md",
    "Egress Traffic Protection": f"{BASE}/security/egress-nat/egress-traffic-protection.md",
    "Distributed Cloud Firewall (DCF)": f"{BASE}/security/dcf/about-dcf-security.md",
    "DCF Egress Rules": f"{BASE}/security/dcf/dcf-create-rules.md",
    "TLS Inspection": f"{BASE}/security/dcf/tls-decryption-configure.md",
    "DNS Hostname SmartGroups": f"{BASE}/resource-groups/smartgroup/hostname-smartgroups-about.md",
    "WebGroups": f"{BASE}/resource-groups/webgroups/webgroups-about.md",
}

security_docs = collect_pages(SECURITY_PAGES)
print(f"\nTotal characters: {len(security_docs)}")

> **Check** — You see one `OK` line per page and a total of a few thousand characters. A `SKIP` line means that page couldn't be downloaded: the others are still used.

### 2.2 The security system prompt

| Choice | Effect |
| --- | --- |
| Role: cloud security architect + network engineer | Explains security from the traffic's point of view, not just as a feature list |
| "Network engineer new to Aviatrix" | Assumes you know BGP and firewalls, but explains Aviatrix's own concepts |
| "Only use information found in the provided pages" | Essential in security: no invented limits or defaults |
| *Sources* after each section | You can check every claim against the original page |
| "How traffic is inspected" as numbered steps | Shows the real path of a packet, the most useful part for a network engineer |
| "Building blocks" table | Lines up DCF, SmartGroups, WebGroups and TLS inspection with their enforcement point |

In [ ]:
security_system_prompt = """
You are a senior cloud security architect and network engineer with deep expertise
in multi-cloud networking (AWS, Azure, GCP), cloud firewalls, egress filtering,
zero-trust segmentation, TLS inspection, BGP and IPsec VPNs.

You receive several pages of the Aviatrix documentation, each marked with
"### SOURCE: <title> (<url>)". Explain how Aviatrix secures cloud traffic
to a network engineer who knows networking but is new to Aviatrix.

Rules:
- Only use information found in the provided pages. Never invent features,
  limits, defaults or configuration values. If something is unclear or missing, say so.
- Use precise terminology (e.g. east-west vs north-south traffic, egress,
  FQDN filtering, SmartGroups, WebGroups, L4 vs L7 rules, policy enforcement point).
- After each section, add the sources you used in the form: *Sources: <title>, <title>*
- Prefer short bullets and tables over long paragraphs.

Structure your answer with these sections:

## Overview
What Aviatrix security does and which problem it solves, in 3 sentences.

## Building blocks
A table with columns: Component | What it does | Where it is enforced.

## How traffic is inspected
The path of a packet through the security controls, as numbered steps.

## Policy model
How rules are built: SmartGroups, WebGroups, rule matching, actions, priorities.

## Egress security
How outbound traffic to the internet is monitored and filtered.

## TLS inspection
Why it is needed, how it works, and the certificate requirements.

## Visibility and monitoring
What you can see and where.

## Prerequisites and limitations
What must be in place first, and what the pages say is not supported.

## Key takeaways
3 bullets a network engineer should remember.

Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

### 2.3 The security user prompt

Unlike Ed's original (*"Provide a short summary of this website"*), this user prompt matches the system prompt:

| Ed's user prompt | This one |
| --- | --- |
| "contents of a website" | Says these are **several documentation pages**, each with a `SOURCE` label |
| "short summary" | Asks for a **thorough** explanation, suitable for studying |
| "summarize news" | Removed: security docs have no news |
| No topic given | Names the 5 topics to focus on |
| — | Tells the model to ignore the index and the Mintlify credit line found in every `.md` page |

In [ ]:
security_user_prompt_prefix = """
Below are several pages of the Aviatrix documentation about security.
Each page starts with "### SOURCE: <title> (<url>)".

Explain how Aviatrix secures cloud traffic, following exactly the structure
in your instructions. Focus on the Distributed Cloud Firewall, egress security,
SmartGroups, WebGroups and TLS inspection.

Be thorough rather than short: this explanation will be used to study the topic.
Ignore any navigation text, page indexes or credit lines that are not documentation.

Documentation:

"""

### 2.4 Explain Aviatrix security

This passes the text from `collect_pages`, not from `fetch_website_contents`: the model receives all eight pages at once.

In [ ]:
explain(security_system_prompt, security_user_prompt_prefix, security_docs)

> **Check** — The answer follows the 9 sections of the system prompt, and each section ends with a *Sources:* line naming the pages it used.

### 2.5 Variations to try

**Answer in Italian, keep technical terms in English.** Add one rule to the system prompt and run the explanation again:

In [ ]:
security_system_prompt_it = security_system_prompt.replace(
    "- Prefer short bullets and tables over long paragraphs.",
    "- Prefer short bullets and tables over long paragraphs.\n- Write in Italian, but keep technical terms in English.",
)

explain(security_system_prompt_it, security_user_prompt_prefix, security_docs)

**Focus on a single topic.** Less material means a more detailed explanation. For example, only the Distributed Cloud Firewall:

In [ ]:
DCF_PAGES = {
    "Distributed Cloud Firewall (DCF)": f"{BASE}/security/dcf/about-dcf-security.md",
    "DCF Egress Rules": f"{BASE}/security/dcf/dcf-create-rules.md",
    "DCF Rule Details": f"{BASE}/security/dcf/dcf-rule-details.md",
    "DCF Monitoring": f"{BASE}/security/dcf/dcf-monitor.md",
    "TLS Inspection": f"{BASE}/security/dcf/tls-decryption-configure.md",
    "Decryption CA Certificates": f"{BASE}/security/dcf/decryption-ca-certificate.md",
}

dcf_docs = collect_pages(DCF_PAGES)
explain(security_system_prompt, security_user_prompt_prefix, dcf_docs)

## Part 3 — Bonus: map the whole documentation with `llms.txt`

Many modern documentation sites publish an `llms.txt` file: a Markdown index of every page, with title, link and a short description, designed to be read by LLMs.
Aviatrix's is at [docs.aviatrix.com/llms.txt](https://docs.aviatrix.com/llms.txt). It's where the security page addresses above came from.

Instead of explaining a few pages, GPT can summarize **the map of the entire documentation**.

In [ ]:
llms_txt = fetch_markdown("https://docs.aviatrix.com/llms.txt", max_chars=15_000)
print(f"Characters: {len(llms_txt)}\n")
print(llms_txt[:1_500])  # show only the beginning

In [ ]:
map_system_prompt = """
You are a tutor helping a network engineer find their way around technical documentation.
You receive the full site index (llms.txt format: title, link, description).
Respond in markdown, without code blocks:
1. In 3 sentences, what the documentation covers as a whole.
2. The main areas it is organized into, one line each.
3. A 5-page reading path for someone starting from scratch, with links.
4. A 5-page reading path focused on security, with links.
"""

map_user_prompt_prefix = "Here is the documentation index:\n\n"

explain(map_system_prompt, map_user_prompt_prefix, llms_txt)

## What I learned

- Ed's pattern (**text → messages → GPT → Markdown**) works on any documentation, not just home pages.
- **Look at the raw text** before calling GPT: it tells you right away whether the problem is the scraper or the prompt.
- Documentation sites often publish **Markdown versions** of their pages (`.md`) and an **`llms.txt`** index: cleaner input, better answers, no scraping headaches.
- **Several short pages, labelled with their source**, give a richer explanation than one truncated page, and let the model cite where each fact comes from.
- The **user prompt must match the system prompt**: Ed's "short summary of a website" would contradict a request for a structured security deep dive.
- In technical and security topics, the rule **"only use information found in the provided pages"** matters most.

**Limits:** each page is truncated to 6,000 characters and `llms.txt` to 15,000, and the model only knows what's in the pages you pass. If Aviatrix moves a page, `collect_pages` reports it as `SKIP`: find the new address in `llms.txt`.

**Possible extensions:** let the user pick the topic (egress, DCF, AgentGuard) as a parameter; compare Aviatrix security with a cloud provider's native controls using their docs; try a free model with Ollama.